# 01 — Data Ingestion & Schema Validation

**Purpose**: Load the raw police violation CSV, validate the schema, cast dtypes, drop excluded columns, and deduplicate.

**Run this notebook cell by cell.** Each cell is independent and prints its own output.

**What you will see**:
- Schema validation results (pass/fail + warnings)
- Dtype cast confirmation
- Null summary for all retained columns
- Deduplication stats
- Final shape and split preview

**Files saved**:
- `data/processed/validation_report.json` — schema validation report
- `data/processed/load_metadata.json` — load stats, file hash, split preview

## Cell 1 — Environment setup
**What this cell does**: Adds the project root to `sys.path` so `src.*` imports work from the notebook.
**Expected output**: No errors. `Project root: ...GridLock R2` printed.

In [1]:
import sys
from pathlib import Path

# Resolve project root (one level above notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'sys.path[0]: {sys.path[0]}')

Project root: C:\Users\USER\Desktop\GridLock_R2_Transfer
sys.path[0]: C:\Users\USER\Desktop\GridLock_R2_Transfer


## Cell 2 — Configure loguru
**What this cell does**: Sets up loguru to print readable log lines in the notebook output.
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='DEBUG',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Schema Validation
**What this cell does**: Calls `validate_schema()` directly on the raw CSV (before any dtype casting).

**Expected output**: `─── Schema Validation PASSED (2 warning(s)) ─────`

Expected warnings (normal):
- 5 unparseable `created_datetime` rows (will be dropped in load.py)
- Leakage columns present (will be dropped in load.py)

If this cell raises `ValueError`, stop and fix before continuing.

In [3]:
import pandas as pd
from src.data.validate import validate_schema, load_eval_config, save_validation_report

CSV_PATH      = PROJECT_ROOT / 'data' / 'raw' / 'jan to may police violation_anonymized791b166.csv'
EVAL_CFG_PATH = PROJECT_ROOT / 'configs' / 'eval.yaml'

print(f'Loading raw CSV from: {CSV_PATH}')
df_raw   = pd.read_csv(CSV_PATH, dtype=str, low_memory=False)
print(f'Raw shape: {df_raw.shape}')

eval_cfg = load_eval_config(EVAL_CFG_PATH)
report   = validate_schema(df_raw, eval_cfg, strict=True)
save_validation_report(report, PROJECT_ROOT / 'data' / 'processed' / 'validation_report.json')

print(f'\nValidation passed: {report["passed"]}')
print(f'Warnings: {len(report["warnings"])}')
print(f'Errors:   {len(report["errors"])}')

Loading raw CSV from: C:\Users\USER\Desktop\GridLock_R2_Transfer\data\raw\jan to may police violation_anonymized791b166.csv
Raw shape: (298450, 24)
21:11:05 | INFO     | Loaded eval config v1.0 from 'C:\Users\USER\Desktop\GridLock_R2_Transfer\configs\eval.yaml'
21:11:05 | INFO     | ─── Schema Validation Start ───────────────────────────────────────
21:11:05 | INFO     | DataFrame shape: 298,450 rows × 24 columns
21:11:05 | INFO     | ✓ All 9 required columns present
21:11:05 | INFO     | ✓ All coordinates within Bengaluru bounding box
21:11:06 | WARNING  | created_datetime: 5 nulls after coerce (≤10 — warning, will be dropped)
21:11:06 | INFO     | ✓ created_datetime range: 2023-11-09 → 2024-04-08 (5 nulls)
21:11:06 | INFO     | ✓ No calendar day gaps in created_datetime
21:11:06 | INFO     | ✓ 'description' confirmed 100% null (expected)
21:11:06 | INFO     | ✓ 'closed_datetime' confirmed 100% null (expected)
21:11:06 | INFO     | ✓ 'action_taken_timestamp' confirmed 100% null (expec

## Cell 4 — Full ingest with load_raw()
**What this cell does**: Calls `load_raw()` — validate + cast dtypes + drop columns + deduplicate.

**Expected output**:
- Progress bars for each step
- `298,450 → 268,281 rows (30,169 exact duplicates removed)`
- `Cols retained: 12`

In [4]:
from src.data.load import load_raw, save_load_metadata

df, metadata = load_raw(
    csv_path=CSV_PATH,
    eval_config_path=EVAL_CFG_PATH,
    save_report=True,
    report_path=PROJECT_ROOT / 'data' / 'processed' / 'validation_report.json',
)
save_load_metadata(metadata, PROJECT_ROOT / 'data' / 'processed' / 'load_metadata.json')

print(f'\nFinal DataFrame shape: {df.shape}')
print(f'Columns retained: {list(df.columns)}')

21:11:06 | INFO     | Reading CSV: 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\raw\jan to may police violation_anonymized791b166.csv' ...


Reading CSV: 100%|██████████| 1/1 [00:02<00:00,  2.39s/file]

21:11:08 | INFO     | Raw CSV loaded: 298,450 rows × 24 columns | SHA-256: 6e4b7ad2e4c713a6...
21:11:08 | INFO     | Loaded eval config v1.0 from 'C:\Users\USER\Desktop\GridLock_R2_Transfer\configs\eval.yaml'
21:11:08 | INFO     | ─── Schema Validation Start ───────────────────────────────────────
21:11:08 | INFO     | DataFrame shape: 298,450 rows × 24 columns
21:11:08 | INFO     | ✓ All 9 required columns present


21:11:09 | INFO     | ✓ All coordinates within Bengaluru bounding box
21:11:09 | WARNING  | created_datetime: 5 nulls after coerce (≤10 — warning, will be dropped)
21:11:09 | INFO     | ✓ created_datetime range: 2023-11-09 → 2024-04-08 (5 nulls)
21:11:09 | INFO     | ✓ No calendar day gaps in created_datetime
21:11:09 | INFO     | ✓ 'description' confirmed 100% null (expected)
21:11:09 | INFO     | ✓ 'closed_datetime' confirmed 100% null (expected)
21:11:09 | INFO     | ✓ 'action_taken_timestamp' confirmed 100% null (expected)
21:11:09 | WARNING  | Leakage columns present — ensure load.py drops them before features: ['data_sent_to_scita_timestamp', 'modified_datetime', 'validation_status', 'validation_timestamp', 'updated_vehicle_number', 'updated_vehicle_type']
21:11:09 | INFO     | ✓ violation_type parseable as list (500-row sample check)
21:11:09 | INFO     | Split preview — train rows (≤2024-02-29): 226,296 | test rows (≥2024-03-01): 70,311
21:11:09 | INFO     | ✓ Temporal split gu

Casting dtypes:   0%|          | 0/4 [00:00<?, ?col-group/s]

...

21:11:09 | INFO     | ✓ Dtypes cast



Dropping excluded columns: 100%|██████████| 1/1 [00:00<00:00, 500.87op/s]

21:11:09 | INFO     | ✓ Dropped 12 excluded columns: ['description', 'closed_datetime', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'modified_datetime', 'validation_status', 'validation_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'id', 'vehicle_number', 'location']
21:11:09 | INFO     |   Retained columns (12): ['latitude', 'longitude', 'vehicle_type', 'violation_type', 'offence_code', 'created_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name']
21:11:09 | INFO     | Null summary (retained columns):
21:11:09 | DEBUG    |   ✓ latitude: 0 nulls
21:11:09 | DEBUG    |   ✓ longitude: 0 nulls
21:11:09 | DEBUG    |   ✓ vehicle_type: 0 nulls
21:11:09 | DEBUG    |   ✓ violation_type: 0 nulls
21:11:09 | DEBUG    |   ✓ offence_code: 0 nulls
21:11:09 | DEBUG    |   ✓ created_datetime: 0 nulls
21:11:09 | DEBUG    |   ✓ device_id: 0 nulls
21:11:09 | DEBUG    |   ✓ created_by_id: 0 nulls
21:11:09 | WARNING  |


Deduplication: 100%|██████████| 2/2 [00:00<00:00, 18.97step/s]

21:11:10 | INFO     | ✓ Deduplication complete: 298,445 → 268,281 rows (30,164 exact duplicates removed)
21:11:10 | INFO     | Split preview → train (≤2024-02-29): 202,968 rows | test (2024-03-01–2024-04-08): 62,583 rows
21:11:10 | INFO     | ─── load_raw() complete: 268,281 rows × 12 columns ───
21:11:10 | INFO     | Load metadata saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\load_metadata.json'

Final DataFrame shape: (268281, 12)
Columns retained: ['latitude', 'longitude', 'vehicle_type', 'violation_type', 'offence_code', 'created_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name']


## Cell 5 — Dtype sanity check
**Expected output**:
- `created_datetime` = `datetime64[ns, UTC]`
- `latitude`, `longitude` = `float64`
- `data_sent_to_scita` = `int8`
- `center_code` ~3.77% nulls (expected — imputed in features.py)

In [5]:
print('=== dtypes ===')
print(df.dtypes)

print('\n=== head(3) ===')
print(df.head(3).to_string())

print('\n=== null counts (retained columns) ===')
null_counts = df.isna().sum()
print(null_counts[null_counts > 0] if null_counts.any() else 'No nulls in retained columns ✓')

print('\n=== violation_type sample ===')
print(df['violation_type'].head(5).tolist())

=== dtypes ===
latitude                          float64
longitude                         float64
vehicle_type                          str
violation_type                        str
offence_code                          str
created_datetime      datetime64[us, UTC]
device_id                             str
created_by_id                         str
center_code                           str
police_station                        str
data_sent_to_scita                   int8
junction_name                         str
dtype: object

=== head(3) ===
    latitude  longitude vehicle_type                                  violation_type offence_code          created_datetime   device_id created_by_id center_code police_station  data_sent_to_scita junction_name
0  12.925557  77.618665          CAR  ["WRONG PARKING","PARKING NEAR ROAD CROSSING"]    [112,104] 2023-11-20 00:28:46+00:00  FKDEV00000    FKUSR00000           9       Madiwala                   1   No Junction
1  12.905463  77.700778     

## Cell 6 — Date range + split preview
**Expected output**:
```
Date range: 2023-11-09 → 2024-04-08
Train rows (≤ 2024-02-29): ~226k
Test rows  (2024-03-01 – 2024-04-08): ~70k
```

In [6]:
import yaml
with open(EVAL_CFG_PATH) as f:
    eval_local = yaml.safe_load(f)
split      = eval_local['split']
train_end  = pd.Timestamp(split['train_end'],  tz='UTC')
test_start = pd.Timestamp(split['test_start'], tz='UTC')
test_end   = pd.Timestamp(split['test_end'],   tz='UTC')

n_train = (df['created_datetime'] <= train_end).sum()
n_test  = ((df['created_datetime'] >= test_start) & (df['created_datetime'] <= test_end)).sum()

print(f"Date range  : {df['created_datetime'].min().date()} → {df['created_datetime'].max().date()}")
print(f"Train rows  : {n_train:,}  (≤ {train_end.date()})")
print(f"Test rows   : {n_test:,}  ({test_start.date()} – {test_end.date()})")
print(f"Dedup removed: {metadata['rows_dropped_dedup']:,} rows")

Date range  : 2023-11-09 → 2024-04-08
Train rows  : 202,968  (≤ 2024-02-29)
Test rows   : 62,583  (2024-03-01 – 2024-04-08)
Dedup removed: 30,164 rows


## Cell 7 — Confirm null columns dropped
**Expected output**: All three show `NOT PRESENT (correctly dropped) ✓`

In [7]:
for col in ['description', 'closed_datetime', 'action_taken_timestamp']:
    status = '⚠  STILL PRESENT' if col in df.columns else '✓  NOT PRESENT (correctly dropped)'
    print(f'  {status}: {col}')

  ✓  NOT PRESENT (correctly dropped): description
  ✓  NOT PRESENT (correctly dropped): closed_datetime
  ✓  NOT PRESENT (correctly dropped): action_taken_timestamp


## Cell 8 — Confirm leakage columns dropped
**Expected output**: All show `NOT PRESENT (correctly excluded) ✓`

In [8]:
for col in ['data_sent_to_scita_timestamp', 'modified_datetime', 'validation_status',
            'validation_timestamp', 'updated_vehicle_number', 'updated_vehicle_type']:
    status = '⚠  STILL PRESENT — leakage risk!' if col in df.columns else '✓  NOT PRESENT (correctly excluded)'
    print(f'  {status}: {col}')

  ✓  NOT PRESENT (correctly excluded): data_sent_to_scita_timestamp
  ✓  NOT PRESENT (correctly excluded): modified_datetime
  ✓  NOT PRESENT (correctly excluded): validation_status
  ✓  NOT PRESENT (correctly excluded): validation_timestamp
  ✓  NOT PRESENT (correctly excluded): updated_vehicle_number
  ✓  NOT PRESENT (correctly excluded): updated_vehicle_type


## Summary

**What was done**:
- Raw CSV loaded and schema-validated (8 checks)
- Dtypes cast: `created_datetime` → UTC datetime64, `latitude`/`longitude` → float64, `data_sent_to_scita` → int8
- 12 excluded columns dropped (null + leakage + identifier)
- Deduplicated by minute-level rule (same-second events at different lat/lon kept)

**What was saved**:
- `data/processed/validation_report.json`
- `data/processed/load_metadata.json`

**Next step**: Run `notebooks/01b_features.ipynb` — feature engineering (Phase A)

In [9]:
print('=== Session Summary ===')
print(f'  Source file     : {metadata["source_file"]}')
print(f'  SHA-256 (16)    : {metadata["file_hash_sha256"][:16]}...')
print(f'  Rows raw        : {metadata["rows_raw"]:,}')
print(f'  Rows after dedup: {metadata["rows_final"]:,}')
print(f'  Cols retained   : {metadata["cols_final"]}')
print(f'  Validation      : {"PASSED" if metadata["validation_passed"] else "FAILED"}')
print(f'  Saved           : data/processed/validation_report.json')
print(f'  Saved           : data/processed/load_metadata.json')
print(f'  Next notebook   : notebooks/01b_features.ipynb')

=== Session Summary ===
  Source file     : C:\Users\USER\Desktop\GridLock_R2_Transfer\data\raw\jan to may police violation_anonymized791b166.csv
  SHA-256 (16)    : 6e4b7ad2e4c713a6...
  Rows raw        : 298,450
  Rows after dedup: 268,281
  Cols retained   : 12
  Validation      : PASSED
  Saved           : data/processed/validation_report.json
  Saved           : data/processed/load_metadata.json
  Next notebook   : notebooks/01b_features.ipynb
